In [1]:
# Mowing Detector in an Irish Field
import openeo

In [2]:
connection = openeo.connect("openeo.dataspace.copernicus.eu/").authenticate_oidc()

Authenticated using refresh token.


In [9]:
import numpy as np
import glob
import rasterio

In [3]:
stac_resource = connection.datacube_from_process(
    "sentinel1_sar_coherence",
    namespace=(
        "https://raw.githubusercontent.com/ESA-APEx/apex_algorithms/"
        "refs/heads/main/algorithm_catalog/eurac/sentinel1_sar_coherence/"
        "openeo_udp/sentinel1_sar_coherence.json"
    ),
    temporal_extent=["2025-05-01", "2025-06-15"],  # spans a suspected mowing window
    temporal_baseline=6,        # days between each coherence pair (6 = S1's minimum)
    burst_id=157107,                 # placeholder — see below, this has to be your real one
    sub_swath="IW2",            # IW1 / IW2 / IW3 — whichever covers your field
    polarization="VH",
    coherence_window_az=2,
    coherence_window_rg=10,
)

In [ ]:
job = stac_resource.create_job(title="TeaG_field_coherence")
job.start_and_wait()
job.get_results().download_files("coherence_output/")

In [11]:
with rasterio.open('coherence_output/openEO_2025-05-03Z.tif') as src:
    coherence = src.read(1)

print("Mean Coherence over field: ", np.nanmean(coherence))

Mean Coherence over field:  0.39896974


In [12]:
coherence_files = sorted(glob.glob("coherence_output/*.tif"))
print(coherence_files)

['coherence_output/openEO_2025-05-03Z.tif', 'coherence_output/openEO_2025-05-09Z.tif', 'coherence_output/openEO_2025-05-15Z.tif', 'coherence_output/openEO_2025-05-21Z.tif', 'coherence_output/openEO_2025-05-27Z.tif', 'coherence_output/openEO_2025-06-02Z.tif', 'coherence_output/openEO_2025-06-08Z.tif']


In [14]:
for f in coherence_files:
    with rasterio.open(f) as src:
        mean_coherence = np.nanmean(src.read(1))
    print(f"{f}: {mean_coherence:3f}")

coherence_output/openEO_2025-05-03Z.tif: 0.398970
coherence_output/openEO_2025-05-09Z.tif: 0.393395
coherence_output/openEO_2025-05-15Z.tif: 0.374605
coherence_output/openEO_2025-05-21Z.tif: 0.327554
coherence_output/openEO_2025-05-27Z.tif: 0.315495
coherence_output/openEO_2025-06-02Z.tif: 0.318379
coherence_output/openEO_2025-06-08Z.tif: 0.334324
